In [ ]:
from src.paths import DATA_ROOT, REPO_ROOT
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F

import os

from PIL import Image
from pathlib import Path
from typing import List, Tuple, Dict

from src.data_preparation.add_noise import add_noise
from torchmetrics.image import StructuralSimilarityIndexMeasure

from scipy import stats
import lpips

# Import the model factory and config
from src.models.model_factory import get_model
from federated_learning_denoising_dentistry.config import MODEL_NAME

In [ ]:
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}


def list_images(folder: Path) -> List[Path]:
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS])

def calculate_patch_positions(image_shape, patch_size=256, max_overlap=24):
    step_size = patch_size - max_overlap
    image_height, image_width = image_shape

    # Calculating the number of patches ensuring they don't exceed the image size
    x_positions = list(range(0, image_width - max_overlap - 1, step_size))
    y_positions = list(range(0, image_height - max_overlap - 1, step_size))

    # Ensure the last patch doesn't exceed the image size by adding positions for any remaining area
    if x_positions[-1] + patch_size > image_width:
        x_positions[-1] = image_width - patch_size
    if y_positions[-1] + patch_size > image_height:
        y_positions[-1] = image_height - patch_size

    # Final patch positions
    patch_positions = [(x, y) for y in y_positions for x in x_positions]

    return patch_positions

def img_stats(image_list) -> Tuple[Dict[Tuple[int, int], int], List[str]]:

    dim_count = {}
    dtype_list = []

    for p in image_list:
        img = Image.open(p)
        img_array = np.array(img)

        dim = img_array.shape
        img_dtype = str(img_array.dtype)

        if dim in dim_count:
            dim_count[dim] += 1
        else:
            dim_count[dim] = 1

        if img_dtype not in dtype_list:
            dtype_list.append(img_dtype)

    return dim_count, dtype_list

In [ ]:
# Move to device if using GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ssim = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)

def psnr(img1, img2):
    
    assert img1.shape == img2.shape, "Images must have the same dimensions"
    # Compute the Mean Squared Error
    #mse = np.mean((img1 - img2) ** 2)
    mse = F.mse_loss(img1, img2, reduction='mean').item()
    if mse == 0:
        return float('inf')
    max_pixel = 1.0  # Assuming the images are normalized between 0 and 1
    return 20 * np.log10(max_pixel / np.sqrt(mse))

def ssim_func(imgs1, imgs2):
    # Convert to grayscale if the images are in color
    
    # Compute SSIM
    ssim_index = ssim(imgs1, imgs2)
    
    return ssim_index

LPIPS_MODEL = lpips.LPIPS(net='alex')

def calculate_lpips(image1, image2): #need to be a 3-channel image
    
    # Initialize LPIPS model
    loss_fn = LPIPS_MODEL
    
    # Convert from [0,1] to [0,255] uint8 format
    if image1.max() <= 1.0:
        image1 = (image1 * 255).astype(np.uint8)
    if image2.max() <= 1.0:
        image2 = (image2 * 255).astype(np.uint8)
    
    # Convert grayscale to 3-channel if needed
    if image1.ndim == 2:
        image1 = np.stack([image1] * 3, axis=-1)
    if image2.ndim == 2:
        image2 = np.stack([image2] * 3, axis=-1)
    
    # Convert images to the required format
    image1_tensor = lpips.im2tensor(image1)
    image2_tensor = lpips.im2tensor(image2)
    # Calculate LPIPS
    lpips_value = loss_fn.forward(image1_tensor, image2_tensor)
    return lpips_value.item()


def compute_gmsd(img1, img2, c=0.0026):
    """
    Gradient Magnitude Similarity Deviation (GMSD) metric.
    Args:
        img1, img2: tensors of shape (B, 1, H, W) or (B, 3, H, W), normalized to [0, 1]
        c: stability constant
    Returns:
        mean GMSD value across batch
    """
    if img1.ndim == 3:
        img1 = img1.unsqueeze(0)
        img2 = img2.unsqueeze(0)
 
    # Convert to grayscale if RGB
    if img1.shape[1] == 3:
        img1 = 0.299 * img1[:,0:1] + 0.587 * img1[:,1:2] + 0.114 * img1[:,2:3]
        img2 = 0.299 * img2[:,0:1] + 0.587 * img2[:,1:2] + 0.114 * img2[:,2:3]
 
    # Sobel filters
    sobel_x = torch.tensor([[1, 0, -1],
                            [2, 0, -2],
                            [1, 0, -1]], dtype=img1.dtype, device=img1.device).unsqueeze(0).unsqueeze(0)
    sobel_y = torch.tensor([[1, 2, 1],
                            [0, 0, 0],
                            [-1, -2, -1]], dtype=img1.dtype, device=img1.device).unsqueeze(0).unsqueeze(0)
 
    # Compute gradients
    grad_x1 = F.conv2d(img1, sobel_x, padding=1)
    grad_y1 = F.conv2d(img1, sobel_y, padding=1)
    grad_x2 = F.conv2d(img2, sobel_x, padding=1)
    grad_y2 = F.conv2d(img2, sobel_y, padding=1)
 
    gm1 = torch.sqrt(grad_x1**2 + grad_y1**2)
    gm2 = torch.sqrt(grad_x2**2 + grad_y2**2)
 
    # Gradient magnitude similarity map
    gms_map = (2 * gm1 * gm2 + c) / (gm1**2 + gm2**2 + c)
 
    # Standard deviation of the similarity map → GMSD
    gmsd_val = torch.std(gms_map, dim=[1, 2, 3])
    return gmsd_val.mean().item()

In [ ]:
# Modified evaluation code with LPIPS and GMSD
# Load all models at once
model_configs = {
    'centralized': f"{DATA_ROOT}/torch_models/centralized_case2/centralized_model_epochXX.pth",
    'fl_model': f"{DATA_ROOT}/torch_models/federated_case3a/federated_model_sim_XXXX_round_XX.pth",
    'local_client0': f"{DATA_ROOT}/torch_models/local_client0_case3a/local_model_epochXX.pth",
    'local_client1': f"{DATA_ROOT}/torch_models/local_client1_case3a/local_model_epochXX.pth",
    'local_client2': f"{DATA_ROOT}/torch_models/local_client2_case3a/local_model_epochXX.pth",
    'local_client3': f"{DATA_ROOT}/torch_models/local_client3_case3a/local_model_epochXX.pth",
    'local_client4': f"{DATA_ROOT}/torch_models/local_client4_case3a/local_model_epochXX.pth",
    'local_client5': f"{DATA_ROOT}/torch_models/local_client5_case3a/local_model_epochXX.pth",
}

test_data_paths = {
    0: f"{DATA_ROOT}/dental_train_val_test_splits_case3a/test/client0",
    1: f"{DATA_ROOT}/dental_train_val_test_splits_case3a/test/client1",
    2: f"{DATA_ROOT}/dental_train_val_test_splits_case3a/test/client2",
    3: f"{DATA_ROOT}/dental_train_val_test_splits_case3a/test/client3",
    4: f"{DATA_ROOT}/dental_train_val_test_splits_case3a/test/client4",
    5: f"{DATA_ROOT}/dental_train_val_test_splits_case3a/test/client5",
    6: f"{DATA_ROOT}/dental_train_val_test_splits_case3a/test/client6",
    7: f"{DATA_ROOT}/dental_train_val_test_splits_case3a/test/client7",
    8: f"{DATA_ROOT}/dental_train_val_test_splits_case3a/test/client8",
    9: f"{DATA_ROOT}/dental_train_val_test_splits_case3a/test/client9",
    10: f"{DATA_ROOT}/dental_train_val_test_splits_case3a/test/client10",
}
patch_size = 256
max_patch_overlap = 20

def denoise_whole_image_with_patches_multi_model(image_path, models, device, patch_size=256, max_overlap=20):
    """
    Denoise a whole image using patch-based inference with multiple models.
    
    Returns:
        dict: Contains original, noisy, and denoised images for each model
    """
    # Load and normalize image
    img = Image.open(image_path)
    img_array = np.array(img)
    
    # Normalize to [0, 1]
    img_dtype = str(img_array.dtype)
    if img_dtype == 'uint8':
        img_array = img_array.astype(np.float32) / 255.0
    elif img_dtype == 'uint16':
        img_array = img_array.astype(np.float32) / 65535.0
    else:
        raise ValueError(f"Unknown image dtype: {img_dtype}")
    
    # Add noise ONCE for all models
    noisy_img_array = add_noise(image=img_array)
    
    # Calculate patch positions
    img_shape = img_array.shape
    patch_positions = calculate_patch_positions(
        image_shape=img_shape, 
        patch_size=patch_size, 
        max_overlap=max_overlap
    )
    
    # Initialize denoised image arrays for each model
    results = {
        'original': img_array,
        'noisy': noisy_img_array,
        'denoised': {},
        'patch_positions': patch_positions
    }
    
    for model_name in models.keys():
        results['denoised'][model_name] = np.zeros_like(img_array)
    
    overlap_count = np.zeros_like(img_array)
    
    # Process each patch with all models
    for x, y in patch_positions:
        # Extract patches
        noisy_patch = noisy_img_array[y:y + patch_size, x:x + patch_size]
        
        # Convert to torch tensor
        if noisy_patch.ndim == 2:
            noisy_tensor = torch.tensor(noisy_patch).unsqueeze(0).unsqueeze(0)
        else:
            noisy_tensor = torch.tensor(noisy_patch).permute(2, 0, 1).unsqueeze(0)
        
        noisy_tensor = noisy_tensor.to(device)
        
        # Run inference with all models
        for model_name, model in models.items():
            with torch.no_grad():
                denoised_tensor = model(noisy_tensor)
            
            # Convert back to numpy
            denoised_patch = denoised_tensor.squeeze().cpu().numpy()
            
            if denoised_patch.ndim == 3 and denoised_patch.shape[0] == 1:
                denoised_patch = denoised_patch.squeeze(0)
            elif denoised_patch.ndim == 3:
                denoised_patch = denoised_patch.transpose(1, 2, 0)
            
            # Accumulate in the denoised image for this model
            results['denoised'][model_name][y:y + patch_size, x:x + patch_size] += denoised_patch
        
        # Count overlaps (same for all models)
        overlap_count[y:y + patch_size, x:x + patch_size] += 1
    
    # Average overlapping regions for all models
    for model_name in models.keys():
        results['denoised'][model_name] = results['denoised'][model_name] / np.maximum(overlap_count, 1)
    
    return results

# Initialize all models
models = {}

for model_name, model_path in model_configs.items():
    print(f"Loading {model_name}...")
    model_state = torch.load(model_path, map_location='cpu')
    model = get_model(MODEL_NAME)
    model.load_state_dict(model_state)
    model.eval()
    model = model.to(device)
    models[model_name] = model
    print(f"✓ {model_name} loaded")

# Initialize list to store raw metrics
raw_metrics_list = []

for client_id, image_dir in test_data_paths.items():
    print(f"\nProcessing Client {client_id}...")
    
    image_list = list_images(Path(image_dir))
    
    for img_idx, p in enumerate(image_list):
        print(f"  Processing image {img_idx + 1}/{len(image_list)}: {p.name}")
        
        # Process whole image with all models
        results = denoise_whole_image_with_patches_multi_model(
            image_path=p,
            models=models,
            device=device,
            patch_size=patch_size,
            max_overlap=max_patch_overlap
        )
        
        # Calculate metrics for each model on the whole reconstructed image
        original_tensor = torch.tensor(results['original']).to(device)
        
        for model_name in models.keys():
            denoised_tensor = torch.tensor(results['denoised'][model_name]).to(device)
            
            # Calculate whole image metrics
            mse_loss = F.mse_loss(denoised_tensor, original_tensor).item()
            psnr_value = psnr(denoised_tensor, original_tensor)
            
            # For SSIM, add batch and channel dimensions
            if original_tensor.ndim == 2:
                original_ssim = original_tensor.unsqueeze(0).unsqueeze(0)
                denoised_ssim = denoised_tensor.unsqueeze(0).unsqueeze(0)
            else:
                original_ssim = original_tensor.permute(2, 0, 1).unsqueeze(0)
                denoised_ssim = denoised_tensor.permute(2, 0, 1).unsqueeze(0)
            
            ssim_value = ssim_func(denoised_ssim, original_ssim).item()
            
            # Calculate LPIPS (on CPU numpy arrays)
            lpips_value = calculate_lpips(
                results['denoised'][model_name], 
                results['original']
            )
            
            # Calculate GMSD
            gmsd_value = compute_gmsd(
                denoised_ssim, 
                original_ssim
            )
            
            # Store raw metrics as a row
            raw_metrics_list.append({
                'model': model_name,
                'testset': client_id,
                'psnr': psnr_value,
                'mse': mse_loss,
                'ssim': ssim_value,
                'lpips': lpips_value,
                'gmsd': gmsd_value
            })

# Convert to DataFrame and save to CSV
metrics_df = pd.DataFrame(raw_metrics_list)

# Define output path
output_csv_path = f"{REPO_ROOT}/notebooks-paper/data/case3a.csv"
metrics_df.to_csv(output_csv_path, index=False)

print(f"\n{'='*70}")
print(f"Raw metrics saved to: {output_csv_path}")
print(f"Total rows: {len(metrics_df)}")
print(f"\nFirst few rows:")
print(metrics_df.head(10))
print(f"\nDataFrame summary:")
print(metrics_df.describe())